# CIFAR-10: One-Change-at-a-Time Experiments

Run all cells in order on a GPU runtime. The original baseline runs for 3 epochs, followed by three 20-epoch comparisons. The exercise cell generates `cifar_results/CIFAR10_Submission.pdf`, plots, logs and a five-sentence analysis based on actual results. Results are empty until execution. In Colab, select a T4 GPU runtime before running.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Hardware Compute Device Selection: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f'Active Compute Device: {device}')

## Dataset Preparation

> **Data Pipeline**: Applying spatial jitter and horizontal flips for training regularization, and normalizing RGB color channels.

In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

import os

data_root = './data'  # ComputerVision/data
archive = os.path.join(data_root, 'cifar-10-python.tar.gz')
extracted = os.path.join(data_root, 'cifar-10-batches-py')

# Use the local CIFAR-10 archive if present; torchvision only downloads when it is missing.
if os.path.isdir(extracted) or os.path.isfile(archive):
    print(f'Using local CIFAR-10 data in {data_root}')
else:
    print('CIFAR-10 not found locally. Downloading...')

train_dataset = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=200, shuffle=False)

print(f'Training Samples: {len(train_dataset)} | Test Samples: {len(test_dataset)}')

In [ ]:
# Visualize sample CIFAR-10 RGB natural images
sample_imgs, sample_lbls = next(iter(train_loader))

def unnormalize_cifar(img_tensor):
    """Reverses normalization for RGB plotting."""
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2023, 0.1994, 0.2010])
    img = img_tensor.permute(1, 2, 0).numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for idx, ax in enumerate(axes):
    img = unnormalize_cifar(sample_imgs[idx])
    lbl = sample_lbls[idx].item()
    ax.imshow(img)
    ax.set_title(classes[lbl], fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle('CIFAR-10 Training Dataset Sample Images & Class Labels', fontsize=12, y=1.08)
plt.tight_layout()
plt.show()

## Model Architecture

> **Model Architecture**: Multi-block VGG/ResNet-inspired convolutional network with Batch Normalization, Dropout, and Global Average Pooling.

In [ ]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(42), num_workers=0)

class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )
        
    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat)

model = CIFAR10CNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'CIFAR-10 CNN Initialized with {total_params:,} parameters.')

## Model Training & Evaluation

> **Training Pipeline**: Multi-epoch optimization with the universal 5-step PyTorch training mechanics and validation tracking.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 3

history = {'train_loss': [], 'test_acc': []}
all_preds = []
all_targets = []

print('Starting CIFAR-10 Training Loop...\n')
for epoch in range(epochs):
    # --- TRAINING PASS ---
    model.train() # 1. Set mode to train
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # STEP 1: Clear previous gradients
        optimizer.zero_grad()
        
        # STEP 2: Forward pass (compute class logits)
        outputs = model(images)
        
        # STEP 3: Compute batch cross-entropy loss
        loss = criterion(outputs, labels)
        
        # STEP 4: Backward pass (compute parameter gradients)
        loss.backward()
        
        # STEP 5: Update network weights
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    history['train_loss'].append(epoch_loss)
    
    # --- EVALUATION PASS ---
    model.eval() # 2. Set mode to eval
    correct = 0
    with torch.inference_mode(): # 3. Disable autograd engine
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            if epoch == epochs - 1:
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(labels.cpu().numpy())
                
    acc = (correct / len(test_loader.dataset)) * 100
    history['test_acc'].append(acc)
    print(f'Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Test Accuracy: {acc:.2f}%')

## Visualizations

> **Visualization**: Plotting the loss convergence, accuracy curves, 10-class Confusion Matrix, and inspecting sample test predictions.

In [ ]:
fig = plt.figure(figsize=(16, 4.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.4])

# Subplot 1: Training Loss Curve
ax1 = fig.add_subplot(gs[0])
ax1.plot(range(1, epochs + 1), history['train_loss'], 'o-', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss Convergence')
ax1.grid(True, linestyle='--', alpha=0.5)

# Subplot 2: Test Accuracy Curve
ax2 = fig.add_subplot(gs[1])
ax2.plot(range(1, epochs + 1), history['test_acc'], 's-', color='#27ae60', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title(f'Test Accuracy (Final: {history["test_acc"][-1]:.2f}%)')
ax2.grid(True, linestyle='--', alpha=0.5)

# Subplot 3: Sample Test Predictions
model.eval()
with torch.inference_mode():
    sample_test_imgs, sample_test_lbls = next(iter(test_loader))
    sample_test_imgs, sample_test_lbls = sample_test_imgs[:6].to(device), sample_test_lbls[:6].to(device)
    sample_test_preds = model(sample_test_imgs).argmax(dim=1)

sub_fig = fig.add_subfigure(gs[2])
sub_axes = sub_fig.subplots(2, 3)
for idx, ax in enumerate(sub_axes.flat):
    img = unnormalize_cifar(sample_test_imgs[idx].cpu())
    true_cls = classes[sample_test_lbls[idx].item()]
    pred_cls = classes[sample_test_preds[idx].item()]
    color = 'green' if true_cls == pred_cls else 'red'
    
    ax.imshow(img)
    ax.set_title(f'Pred: {pred_cls}\n(True: {true_cls})', color=color, fontsize=8, fontweight='bold')
    ax.axis('off')
sub_fig.suptitle('Sample Test Predictions on Natural Images', fontsize=11)

plt.tight_layout()
plt.show()

## Summary

- **Color Multi-Channel Convolutions**: Processing 3-channel RGB tensors with receptive fields extending across depth channels.
- **Data Augmentation**: Spatial flips and random crops prevent overfitting on small natural image datasets.
- **The Universal 5 Steps**: (1) `optimizer.zero_grad()` $\rightarrow$ (2) `outputs = model(images)` $\rightarrow$ (3) `loss = criterion(...)` $\rightarrow$ (4) `loss.backward()` $\rightarrow$ (5) `optimizer.step()`.

## Exercise: Improve the Test Accuracy

The baseline network above is small (about 111k parameters) and trains for only 3 epochs, so its test accuracy is far from what CIFAR-10 allows. **Your goal: raise the test accuracy as high as you can** (a good target is **80%+**; well-tuned CNNs reach 90%+).

**Rules**
- Train only on the CIFAR-10 training set. Do not train on the test set.
- Do not use pretrained weights.
- Record your baseline accuracy first, then change **one thing at a time** so you know what helped.

**Ideas to try**
1. **Train longer**: increase `epochs` (try 15-30).
2. **Learning rate schedule**: add `optim.lr_scheduler.OneCycleLR` or `CosineAnnealingLR`.
3. **Wider / deeper network**: more channels (64-128-256), a third conv block, two convs per block.
4. **Regularization**: tune dropout, add weight decay (`optim.AdamW(..., weight_decay=5e-4)`).
5. **Stronger augmentation**: `ColorJitter`, `RandomErasing`, or `TrivialAugmentWide`.
6. **Optimizer / batch size**: compare Adam vs. SGD with momentum (0.9), try batch sizes 128-256.
7. **Residual connections**: add skip connections as in a small ResNet.

**Deliverable**: fill in the starter cell below, then write 3-5 sentences answering:
- Which change gave the biggest improvement, and why do you think it helped?
- Did anything you tried hurt accuracy or cause overfitting? How can you tell from the train loss and test accuracy curves?
- Which classes are still confused most (see the confusion matrix)?

In [ ]:
# Run after the baseline cells. All runs use only CIFAR-10 train=True for updates.
import random, json, copy
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

SEED = 42
OUT = Path('cifar_results'); OUT.mkdir(exist_ok=True)
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def loaders():
    # Reset the shuffle generator for a fair fresh-start comparison.
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g, num_workers=0)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels))
    def forward(self, x): return torch.relu(x + self.net(x))

class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(),nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3,64), ResidualBlock(64),
            block(64,128), ResidualBlock(128), block(128,256), ResidualBlock(256), nn.Dropout(.25))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(256,128),nn.ReLU(),nn.Dropout(.5),nn.Linear(128,10))
    def forward(self,x): return self.classifier(self.features(x))

@torch.inference_mode()
def evaluate(net):
    net.eval(); pred=[]; target=[]
    for x,y in test_loader:
        pred.extend(net(x.to(device)).argmax(1).cpu().tolist());target.extend(y.tolist())
    return 100*np.mean(np.array(pred)==np.array(target)), pred, target

def train_experiment(name, factory, epochs, scheduled=False):
    seed_all(); net=factory().to(device); loader=loaders()
    optimizer=optim.Adam(net.parameters(), lr=.001)
    # The schedule is the only optimizer difference from the preceding run.
    scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs,eta_min=1e-5) if scheduled else None
    criterion=nn.CrossEntropyLoss(); h={'train_loss':[],'test_acc':[]}; lines=[]
    print('\n'+name, '| parameters:',sum(p.numel() for p in net.parameters()),flush=True)
    for epoch in range(epochs):
        net.train(); total=0.
        for images,labels in loader:
            images,labels=images.to(device),labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs=net(images)
            loss=criterion(outputs,labels)
            loss.backward()
            optimizer.step()
            total+=loss.item()*images.size(0)
        if scheduler: scheduler.step()
        acc,pred,target=evaluate(net)
        h['train_loss'].append(total/len(loader.dataset));h['test_acc'].append(float(acc))
        line=f"Epoch [{epoch+1}/{epochs}] | Loss: {h['train_loss'][-1]:.4f} | Test Accuracy: {acc:.2f}%"
        lines.append(line); print(line,flush=True)
        # Persist each epoch, including the epoch-3 baseline before further training.
        (OUT/(name+'_history.json')).write_text(json.dumps(h,indent=2))
        (OUT/(name+'_output.txt')).write_text('\n'.join(lines))
        torch.save({'model':net.state_dict(),'optimizer':optimizer.state_dict(),'epoch':epoch+1},OUT/(name+'_last.pt'))
    return {'name':name,'history':h,'line':lines[-1],'pred':pred,'target':target,
            'params':sum(p.numel() for p in net.parameters())}

# Baseline was recorded above, before any model modification.
assert len(history['test_acc']) == 3, 'Run the original 3-epoch baseline first.'
baseline_accuracy=float(history['test_acc'][-1])
print(f'Original baseline (3 epochs): {baseline_accuracy:.2f}%')
results=[{'name':'A_baseline','history':copy.deepcopy(history),
    'line':f"Epoch [3/3] | Loss: {history['train_loss'][-1]:.4f} | Test Accuracy: {baseline_accuracy:.2f}%",
    'pred':list(map(int,all_preds)), 'target':list(map(int,all_targets)), 'params':111498}]
# Each comparison changes ONE factor. Do not change batch size or augmentation.
results.append(train_experiment('B_longer', CIFAR10CNN, 20, False))
results.append(train_experiment('C_schedule', CIFAR10CNN, 20, True))
results.append(train_experiment('D_residual_architecture', ImprovedCNN, 20, True))

# Compare final epochs, not the highest epoch along each test curve.
# This is exploratory test-set comparison as requested, not an unbiased final audit.
best=max(results,key=lambda r:r['history']['test_acc'][-1])
print('\nBASELINE:',results[0]['line'])
print('BEST CONFIGURATION:',best['name'])
print('FINAL TEST OUTPUT:',best['line'])
print('All models initialized from scratch; no pretrained weights.')
for r in results: print(r['name'],f"{r['history']['test_acc'][-1]:.2f}%")

cm=confusion_matrix(best['target'],best['pred'],labels=list(range(10)))
assert cm.sum()==10000
fig_cm,ax=plt.subplots(figsize=(10,8))
ConfusionMatrixDisplay(cm,display_labels=classes).plot(ax=ax,cmap='Blues',colorbar=False,values_format='d')
ax.set_title('Final-epoch confusion matrix: '+best['name']);plt.tight_layout()
fig_cm.savefig(OUT/'confusion_matrix.png',dpi=180,bbox_inches='tight');plt.show()

fig_curves,axes=plt.subplots(1,2,figsize=(12,4))
for r in results:
    h=r['history'];x=range(1,len(h['train_loss'])+1)
    axes[0].plot(x,h['train_loss'],label=r['name']);axes[1].plot(x,h['test_acc'],label=r['name'])
for ax in axes: ax.set_xlabel('Epoch');ax.grid(alpha=.25);ax.legend(fontsize=7)
axes[0].set(title='Training loss',ylabel='Cross entropy');axes[1].set(title='Test accuracy',ylabel='Accuracy (%)')
fig_curves.tight_layout();fig_curves.savefig(OUT/'training_curves.png',dpi=180,bbox_inches='tight');plt.show()

# Derive the note from observed results, never hard-code an accuracy or confusion pair.
accs=[r['history']['test_acc'][-1] for r in results]
deltas=np.diff(accs); changes=['training for 20 instead of 3 epochs','adding cosine learning-rate decay','switching to the wider residual architecture']
k=int(np.argmax(deltas))
pairs=sorted([(int(cm[i,j]+cm[j,i]),i,j) for i in range(10) for j in range(i+1,10)],reverse=True)[:2]
confused=' and '.join(f'{classes[i]}/{classes[j]} ({v} combined errors)' for v,i,j in pairs)
if deltas[k]>0: first=f'The largest adjacent improvement was {changes[k]}, adding {deltas[k]:.2f} percentage points; this suggests that this change helped under the fixed experimental conditions.'
else: first='None of the tested changes improved final test accuracy over its preceding configuration.'
hurts=[f'{changes[i]} ({d:+.2f} points)' for i,d in enumerate(deltas) if d<0]
second=('Changes that reduced final accuracy were '+', '.join(hurts)+'.') if hurts else 'None of the adjacent changes reduced final test accuracy.'
h=best['history']; over=len(h['train_loss'])>=6 and h['train_loss'][-1]<h['train_loss'][-6] and h['test_acc'][-1]<h['test_acc'][-6]
third='Over the final five-epoch interval, training loss fell while test accuracy also fell, consistent with possible overfitting.' if over else 'The final five-epoch interval does not show the simple pattern of falling training loss accompanied by falling test accuracy, although this alone cannot rule out overfitting.'
note=' '.join([first,second,third,'The most confused class pairs were '+confused+'.','These are single-seed exploratory comparisons using test curves; repeated runs and a training-only validation split would strengthen the conclusions.'])
print('\n'+note)
(OUT/'analysis.txt').write_text(note)

# One submission PDF: exact printed final lines, confusion matrix and curves.
# PNG exports are direct figures from this execution, not fabricated UI screenshots.
fig_summary=plt.figure(figsize=(11.7,8.3));fig_summary.suptitle('CIFAR-10 Accuracy Improvement | Yan Zhou',fontsize=18,y=.94)
lines=['Baseline — '+results[0]['line'], 'Best model — '+best['name'],best['line'],'']
lines += [f"{r['name']}: {r['history']['test_acc'][-1]:.2f}% | {r['params']:,} parameters" for r in results]
fig_summary.text(.07,.84,'\n'.join(lines),fontfamily='monospace',fontsize=11,va='top')
fig_summary.text(.07,.48,'\n'.join(textwrap.wrap(note,110)),fontsize=11,va='top',linespacing=1.6)
fig_summary.text(.07,.08,'Training data: CIFAR-10 training set only (50,000). Test: 10,000. No pretrained weights.',fontsize=10)
fig_summary.savefig(OUT/'final_accuracy_and_note.png',dpi=180,bbox_inches='tight')
with PdfPages(OUT/'CIFAR10_Submission.pdf') as pdf:
    pdf.savefig(fig_summary,bbox_inches='tight');pdf.savefig(fig_cm,bbox_inches='tight');pdf.savefig(fig_curves,bbox_inches='tight')
plt.show()
print('Created:',(OUT/'CIFAR10_Submission.pdf').resolve())
# If the instructor requires literal notebook screenshots, capture the displayed
# final-output block, confusion matrix and curves above; all values match the PDF.
try:
    from IPython.display import FileLink,display
    display(FileLink(str(OUT/'CIFAR10_Submission.pdf')))
except ImportError: pass


The exercise cell prints the analysis and writes a single PDF containing actual final epoch outputs and result figures.